# NeurIPS 2026 Smart Buildings — Submission Version 11

**Experiment:** Reheat-aware supply-air-temperature (SAT) reset for AHUs with mixed heating and cooling demand. V11 explores an actuator and cost coupling that earlier versions have not directly targeted: when warm and cold zones share an AHU, deliberately warm the cooling-coil command to avoid overcooling zones that then require VAV reheat, **subject to a hard cooling-headroom guard for each hot zone**.

## Algorithm and responsibilities

| Component | Purpose |
| --- | --- |
| `_build_topology` | Discover each AHU's zones and action indices from metadata |
| `_zone_demand`, `_comfort_fallback` | Decode normalized zone temperature and validated comfort bounds |
| `_choose_supply_temperature` | Compute cooling baseline; apply heating-share SAT reset; protect hot zones |
| `_set_terminal` | Adjust cooling dampers and reheat valves for the selected SAT |
| `_control_ahu` | Coordinate one AHU with a fixed active fan pressure (850 Pa) |
| `_control_hot_water` | Start and set the central hot-water plant only for valid reheat requests |
| `_idle_actions`, `reset`, `get_action` | Honor the normalized Kaggle contract; keep episodes stateless |

The heat/cool load ratio is computed **within each AHU**, not across the building; a cold AHU cannot suppress cooling in another AHU. Peak cooling sets the baseline SAT (297 K minus 1.9 K per kelvin of cooling error); capped zone heating share may add up to 5 K. A guarded cooling zone caps the result below its current temperature, with at least 0.4 K headroom for mild violations and greater headroom for significant violations. Heating-only AHUs command the 305 K cooling-coil idle setpoint, not physical heating.

V11 does **not** reuse V2/V3's PID, V4's future-temperature forecast, V5's regret allocation, V6's RLS candidate planner, V7's economizer/mixing physics, V8's time-domain hysteresis, V9's building occupancy-weighted activation, or V10's fan-pressure trim-and-respond. Constant active fan pressure isolates the SAT hypothesis and is **not** asserted to be optimal. Complexity is O(AHUs + zones) with constant additional per-zone work and no episode state.

## Known limitations and validation

This is a **research experiment**, not a proven score improvement. Temperature-headroom and damper mappings are heuristic; there is no thermal simulator or direct reward optimization inside the policy. The delivered AHU supply temperature may differ from the coil command, especially under cold outside air. Sensor availability and action normalization must generalize from public metadata.

**Synthetic contract/regression tests:** `python Co_Kaggle/g8/submissions/test_version11.py`; the test executes this notebook's actual policy and gateway cells. Full paired 30-scenario public starter-kit evaluation, then Kaggle scoring, are required for performance claims. The recorded best prior public score is V7: **0.03947** (lower is better).

**Ablation:** Compare V11 as shipped against an otherwise identical version with `MAX_REHEAT_AVOIDANCE_RESET_K = 0` to isolate reheat-aware SAT reset; track comfort, gas, electricity, carbon, and per-building/weather scores.


In [ ]:
"""Version 11: reheat-aware supply-air-temperature reset.

A coupled, instantaneous AHU controller: supply air is warmed when heating and
cooling zones share one AHU, subject to per-hot-zone cooling-headroom guards.
The algorithm uses present zone demand only, without learning, PID, future
forecasts, pressure trim/response or a search over candidate plant actions.

The cooling coil cannot heat. A 305 K coil command merely disables additional
mechanical cooling; VAV reheat and the central hot-water plant serve cold zones.
Temperature units are kelvin; output actions lie in [-1, 1].
"""
import re

import numpy as np
from kaggle_evaluation import smart_buildings_inference_server


class TransferPolicy(smart_buildings_inference_server.Policy):
    """Metadata-driven instantaneous reheat-aware AHU supply reset policy."""

    # Run only for meaningful zone errors. Five-minute policy step.
    START_CONDITIONING_ERROR_K = 0.25

    # Mechanical cooling is disabled at the upper supply-air command (305 K).
    MAX_SUPPLY_K = 305.0
    MIN_SUPPLY_K = 285.0
    COOLING_BASE_K = 297.0
    COOLING_GAIN_K_PER_ERROR_K = 1.9

    # A common AHU serving both warm and cold zones may accept warmer air.
    # This soft preference is bounded by the guard below, not by hard modes.
    MAX_REHEAT_AVOIDANCE_RESET_K = 5.0
    LOAD_CAP_K = 3.0
    HEADROOM_GUARD_START_K = 0.25
    STRONG_COOLING_START_K = 1.5
    MIN_COOLING_HEADROOM_K = 0.4
    STRONG_COOLING_HEADROOM_K = 2.0
    STRONG_COOLING_HEADROOM_GAIN = 0.75

    # Keep fan pressure fixed in this experiment to isolate supply reset.
    # The fan is only powered when there is a conditioning request.
    ACTIVE_FAN_PRESSURE_PA = 850.0

    def __init__(self, building_meta_data, action_spec):
        super().__init__(building_meta_data, action_spec)
        metadata = building_meta_data
        self.continuous_count = int(metadata["n_continuous_action"])
        self.discrete_count = int(metadata["n_discrete_action"])
        self.observation_index = {
            str(name): position
            for position, name in enumerate(metadata["observation_field_names"])
        }
        self.supply_indices = list(map(int, metadata["temp_indices"]))
        self.fan_indices = list(map(int, metadata["pressure_indices"]))
        self.recirc_indices = list(map(int, metadata["recirculation_indices"]))
        water_indices = list(map(int, metadata["water_indices"]))
        self.ahu_count = len(self.supply_indices)
        self.water_run_index = self.discrete_count - 1
        self.water_supply_index = water_indices[0] if water_indices else None
        self.water_pressure_index = (
            self.fan_indices[self.ahu_count]
            if len(self.fan_indices) > self.ahu_count else None
        )

        self.comfort_band = tuple(map(float, metadata["comfort_temp_window"]))
        self.eco_band = tuple(map(float, metadata["eco_temp_window"]))
        stats = metadata.get("observation_normalization_metadata", {}).get(
            "zone_air_temperature_sensor", {}
        )
        self.zone_mean = float(stats.get("sample_mean", 295.0))
        self.zone_std = float(stats.get("sample_std", 1.0))
        if not np.isfinite(self.zone_mean):
            self.zone_mean = 295.0
        if not np.isfinite(self.zone_std) or self.zone_std <= 0:
            self.zone_std = 1.0

        self.ahus = self._build_topology(metadata)
        self.reset()

    # --- 1. Topology: resolve dynamic field/action indices once per building. ---
    @staticmethod
    def _numeric_order(name):
        match = re.search(r"\d+", str(name))
        return (int(match.group()) if match else -1, str(name))

    def _field(self, name):
        return self.observation_index.get(name)

    def _build_topology(self, metadata):
        """Build AHU -> VAV mappings using the public building metadata."""
        ahus = []
        assignments = metadata["ahu_zone_assignments"]
        zone_actions = metadata["zone_action_indices"]
        for ahu_id, ahu_name in enumerate(sorted(assignments, key=self._numeric_order)):
            if ahu_id >= self.ahu_count:
                break
            zones = []
            for room in assignments[ahu_name]:
                vav_name = str(room).replace("room_", "VAV_", 1)
                temperature_index = self._field(
                    f"{vav_name}_zone_air_temperature_sensor"
                )
                if temperature_index is None:
                    continue
                actions = zone_actions.get(vav_name, {})
                zones.append({
                    "temperature": temperature_index,
                    "heat_setpoint": self._field(
                        f"{vav_name}_zone_air_heating_temperature_setpoint_sensor"
                    ),
                    "cool_setpoint": self._field(
                        f"{vav_name}_zone_air_cooling_temperature_setpoint_sensor"
                    ),
                    "damper_action": actions.get("supply_air_damper_percentage_command"),
                    "reheat_action": actions.get("reheat_valve_percentage_command"),
                })
            ahus.append({
                "id": ahu_id,
                "supply_action": self.supply_indices[ahu_id],
                "fan_action": (
                    self.fan_indices[ahu_id] if ahu_id < len(self.fan_indices) else None
                ),
                "recirc_action": (
                    self.recirc_indices[ahu_id]
                    if ahu_id < len(self.recirc_indices) else None
                ),
                "zones": zones,
            })
        return ahus

    # --- 2. Sensor decoding and normalised-action encoding. ---
    @staticmethod
    def _clamp(value, low, high):
        return float(np.clip(value, low, high))

    @classmethod
    def _fraction_action(cls, opening):
        """Map a physical [0, 1] opening to the required [-1, 1] action."""
        return 2.0 * cls._clamp(opening, 0.0, 1.0) - 1.0

    @classmethod
    def _supply_action(cls, kelvin):
        """Supply setpoint 285..305 kelvin -> normalized [-1, 1]."""
        return cls._clamp((kelvin - 295.0) / 10.0, -1.0, 1.0)

    @staticmethod
    def _read(observation, index):
        if index is None or not 0 <= index < len(observation):
            return float("nan")
        return float(observation[index])

    def _write(self, actions, index, value):
        if index is not None and 0 <= index < len(actions):
            actions[index] = self._clamp(value, -1.0, 1.0)

    def _comfort_fallback(self, observation):
        """Select the published comfort/eco band if live setpoints are invalid."""
        comfort_now = self._read(
            observation, self._field("comfort_mode_now")
        )
        return (self.comfort_band if np.isfinite(comfort_now) and comfort_now >= 0.5
                else self.eco_band)

    def _zone_demand(self, observation, zone, fallback):
        """Return heating/cooling shortfall in K, or None for bad temperature."""
        normalized = self._read(observation, zone["temperature"])
        temperature_k = self.zone_mean + self.zone_std * normalized
        if not np.isfinite(temperature_k) or not 240 <= temperature_k <= 340:
            return None
        heat_setpoint = self._read(observation, zone["heat_setpoint"])
        cool_setpoint = self._read(observation, zone["cool_setpoint"])
        if not (np.isfinite(heat_setpoint) and np.isfinite(cool_setpoint)
                and 245 <= heat_setpoint < cool_setpoint <= 335
                and cool_setpoint - heat_setpoint <= 35):
            heat_setpoint, cool_setpoint = fallback
        return {
            "zone": zone,
            "temperature_k": temperature_k,
            "heating_error": max(0.0, heat_setpoint - temperature_k),
            "cooling_error": max(0.0, temperature_k - cool_setpoint),
        }

    # --- 3. Coupled supply reset: avoid reheat without losing cooling. ---
    def _choose_supply_temperature(self, demands):
        """Select the warmest useful supply temperature with a cooling guard.

        The first term cools according to the worst hot zone. If cold zones
        share the AHU, the coil setpoint is raised proportionally to their
        capped demand. A hard per-zone headroom cap prevents this preference
        from sacrificing significant cooling demands.

        The decision is O(zones), stateless and depends only on sensors
        available in the current policy invocation. It is not MPC.
        """
        peak_cooling_k = max(d["cooling_error"] for d in demands)
        if peak_cooling_k < self.START_CONDITIONING_ERROR_K:
            return self.MAX_SUPPLY_K

        cooling_base_k = self._clamp(
            self.COOLING_BASE_K - self.COOLING_GAIN_K_PER_ERROR_K * peak_cooling_k,
            self.MIN_SUPPLY_K,
            self.MAX_SUPPLY_K,
        )
        total_heating = sum(
            min(d["heating_error"], self.LOAD_CAP_K) for d in demands
        )
        total_cooling = sum(
            min(d["cooling_error"], self.LOAD_CAP_K) for d in demands
        )
        heating_share = total_heating / max(1e-6, total_heating + total_cooling)
        candidate_k = (
            cooling_base_k
            + self.MAX_REHEAT_AVOIDANCE_RESET_K * heating_share
        )

        for demand in demands:
            error_k = demand["cooling_error"]
            if error_k < self.HEADROOM_GUARD_START_K:
                continue
            if error_k < self.STRONG_COOLING_START_K:
                headroom_k = self.MIN_COOLING_HEADROOM_K
            else:
                headroom_k = (
                    self.STRONG_COOLING_HEADROOM_K
                    + self.STRONG_COOLING_HEADROOM_GAIN
                    * min(error_k, 5.0)
                )
            # Hot zones must receive air cooler than their own temperature.
            candidate_k = min(
                candidate_k, demand["temperature_k"] - headroom_k
            )

        return self._clamp(
            candidate_k, self.MIN_SUPPLY_K, self.MAX_SUPPLY_K
        )

    # --- 4. VAV terminal commands and shared reheat requests. ---
    def _set_terminal(self, demand, supply_k, actions, heating_requests):
        """Use the selected supply temperature to coordinate VAV/reheat."""
        heating_k = demand["heating_error"]
        cooling_k = demand["cooling_error"]
        zone = demand["zone"]
        damper_fraction = 0.10
        valve_fraction = 0.0

        if cooling_k >= self.START_CONDITIONING_ERROR_K:
            # Warm reset gives less cooling per unit of air, so allow a
            # larger damper command. This is an airflow proxy, not a model.
            warm_supply_adjustment = 0.035 * max(0.0, supply_k - 293.0)
            damper_fraction = self._clamp(
                0.18 + 0.28 * cooling_k + warm_supply_adjustment, 0.18, 1.0
            )
        elif heating_k >= self.START_CONDITIONING_ERROR_K:
            damper_fraction = self._clamp(
                0.28 + 0.20 * heating_k, 0.28, 1.0
            )
            reheat_index = zone["reheat_action"]
            if reheat_index is not None and 0 <= reheat_index < self.continuous_count:
                # Colder supply needs extra heat from a terminal coil.
                cold_supply_adjustment = 0.018 * max(0.0, 295.0 - supply_k)
                valve_fraction = self._clamp(
                    0.18 + 0.27 * heating_k + cold_supply_adjustment,
                    0.18, 1.0
                )
                heating_requests.append((heating_k, valve_fraction))

        self._write(
            actions, zone["damper_action"], self._fraction_action(damper_fraction)
        )
        self._write(
            actions, zone["reheat_action"], self._fraction_action(valve_fraction)
        )

    # --- 5. AHU dispatch: a fixed fan isolates the new supply reset. ---
    def _control_ahu(self, ahu, demands, actions, switches, heating_requests):
        """Run conditioning when any valid zone is outside the control band."""
        if not demands:
            return
        peak_error_k = max(
            max(d["heating_error"], d["cooling_error"]) for d in demands
        )
        if peak_error_k < self.START_CONDITIONING_ERROR_K:
            return

        switches[ahu["id"]] = 1
        supply_k = self._choose_supply_temperature(demands)
        self._write(
            actions, ahu["supply_action"], self._supply_action(supply_k)
        )
        self._write(actions, ahu["recirc_action"], 1.0)
        self._write(
            actions,
            ahu["fan_action"],
            self.ACTIVE_FAN_PRESSURE_PA / 1000.0 - 1.0,
        )
        for demand in demands:
            self._set_terminal(demand, supply_k, actions, heating_requests)

    def _control_hot_water(self, heating_requests, actions, switches):
        """Only start the central water loop for valid VAV reheat requests."""
        if not heating_requests or not 0 <= self.water_run_index < self.discrete_count:
            return
        switches[self.water_run_index] = 1
        worst_error = max(error for error, _ in heating_requests)
        mean_valve = sum(valve for _, valve in heating_requests) / len(heating_requests)
        water_k = self._clamp(323.0 + 4.0 * worst_error, 320.0, 348.0)
        self._write(actions, self.water_supply_index,
                    self._fraction_action((water_k - 310.0) / 40.0))
        self._write(actions, self.water_pressure_index,
                    self._fraction_action(self._clamp(0.40 + 0.50 * mean_valve,
                                                       0.40, 0.95)))

    def _idle_actions(self):
        """Normalized zero means half-open, not off: choose explicit idle."""
        actions = np.full(self.continuous_count, -1.0, dtype=np.float32)
        switches = np.zeros(self.discrete_count, dtype=np.int32)
        for index in self.supply_indices:
            self._write(actions, index, 1.0)  # Idle cooling coil.
        for index in self.recirc_indices:
            self._write(actions, index, 1.0)  # Maximum permitted return air.
        return actions, switches

    def reset(self):
        """No per-episode state: identical observations yield identical actions."""
        return None

    def get_action(self, obs):
        """One O(AHUs + zones), stateless five-minute control step."""
        observation = np.asarray(obs, dtype=np.float64).reshape(-1)
        actions, switches = self._idle_actions()
        fallback = self._comfort_fallback(observation)
        heating_requests = []

        for ahu in self.ahus:
            demands = [self._zone_demand(observation, z, fallback)
                       for z in ahu["zones"]]
            valid_demands = [d for d in demands if d is not None]
            self._control_ahu(ahu, valid_demands,
                              actions, switches, heating_requests)

        self._control_hot_water(heating_requests, actions, switches)
        np.nan_to_num(actions, copy=False, nan=-1.0, posinf=1.0, neginf=-1.0)
        np.clip(actions, -1.0, 1.0, out=actions)
        return {"continuous_action": actions, "discrete_action": switches}


In [ ]:
# Official Kaggle notebook inference gateway.
smart_buildings_inference_server.SmartBuildingsInferenceServer(TransferPolicy).run()
